# 🔬 Product Search Experiment: Preprocessing, ChromaDB VectorDB & BM25 RRF Hybrid Search

สมุดโน้ตเล่มนี้จัดทำขึ้นเพื่อ **เตรียมข้อมูล (Text Preprocessing) จาก SQLite Database (`yuedpao_chatbot.db`)** และทดสอบระบบค้นหาสินค้าแบบ **Hybrid Search (BM25 + ChromaDB Vector Store)** ผสานด้วย **Reciprocal Rank Fusion (RRF)** พร้อม **ชุดทดสอบและวัดผลลัพธ์ QA Benchmark Dataset 50 คำถาม**

## 🛠️ Step 1: โหลดไลบรารีและดึงข้อมูลสินค้าจาก SQLite (`yuedpao_chatbot.db`)

In [41]:
import sqlite3
import os
import sys
import json
import re
import time
import numpy as np
from typing import List, Dict, Any

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

db_path = os.path.join("..", "..", "yuedpao_chatbot.db")
if not os.path.exists(db_path):
    db_path = os.path.join("..", "yuedpao_chatbot.db")
if not os.path.exists(db_path):
    db_path = "yuedpao_chatbot.db"

print(f"📂 เชื่อมต่อฐานข้อมูล: {db_path}")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
SELECT product_id, name, category, fabric_collection, style_fit, price, description, image_url 
FROM products
""")
product_rows = cursor.fetchall()

cursor.execute("SELECT product_id, GROUP_CONCAT(DISTINCT color_name) FROM product_variants GROUP BY product_id")
variant_color_map = dict(cursor.fetchall())

products = []
for r in product_rows:
    p_id = r[0]
    colors_str = variant_color_map.get(p_id, "") or ""
    products.append({
        "id": p_id, "name": r[1], "category": r[2], "fabric": r[3],
        "style": r[4], "price": r[5], "description": r[6] or "",
        "image_url": r[7] or "", "colors": colors_str
    })

conn.close()
print(f"✅ ดึงข้อมูลสินค้าสำเร็จ! ทั้งหมด {len(products):,} รายการ")

📂 เชื่อมต่อฐานข้อมูล: ..\..\yuedpao_chatbot.db
✅ ดึงข้อมูลสินค้าสำเร็จ! ทั้งหมด 695 รายการ


## 🧹 Step 2: ฟังก์ชันทำความสะอาดข้อความ (Text Preprocessing & Cleansing)

In [42]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"ส่งฟรี\*?", "", text)
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if products:
    sample_p = products[0]
    print("🔸 ก่อนคลีน:", repr(sample_p['category']))
    print("🔹 หลังคลีน :", repr(clean_text(sample_p['category'])))

🔸 ก่อนคลีน: 'RUNNING ROULETTE'
🔹 หลังคลีน : 'RUNNING ROULETTE'


## 📝 Step 3: สร้าง Rich Composite Documents (`passage: ...`)

In [43]:
documents = []
doc_ids = []
metadatas = []

for p in products:
    clean_name = clean_text(p["name"])
    clean_cat  = clean_text(p["category"])
    clean_desc = clean_text(p["description"])
    colors_info = f"สี: {p['colors']}" if p["colors"] else ""

    doc_text = (
        f"passage: สินค้า: {clean_name} | หมวดหมู่: {clean_cat} | "
        f"เทคโนโลยีผ้า: {p['fabric']} | ทรงเสื้อ: {p['style']} | ราคา: ฿{p['price']} | "
        f"{colors_info} | รายละเอียดและจุดเด่น: {clean_desc}"
    )
    documents.append(doc_text)
    doc_ids.append(f"prod_{p['id']}")
    metadatas.append({
        "product_id": p["id"], "name": clean_name, "category": clean_cat,
        "fabric": p["fabric"], "style": p["style"],
        "price": p["price"], "image_url": p["image_url"]
    })

print(f"✅ สร้าง {len(documents):,} Rich Composite Documents เรียบร้อย!")
print(f"\nตัวอย่าง Document [1]:\n{documents[0]}")

✅ สร้าง 695 Rich Composite Documents เรียบร้อย!

ตัวอย่าง Document [1]:
passage: สินค้า: Running Roulette_Dark Gray Bleached | หมวดหมู่: RUNNING ROULETTE | เทคโนโลยีผ้า: Classic Cotton | ทรงเสื้อ: Unisex | ราคา: ฿390 | สี: Dark Gray | รายละเอียดและจุดเด่น: New Collection! RUNNING ROULETTE🏃‍♂️🔥เสื้อฟอกทรงโอเวอร์ไซ...


## 🤖 Step 4: โหลดโมเดล Embedding `intfloat/multilingual-e5-small`

In [44]:
from sentence_transformers import SentenceTransformer

print("⏳ กำลังโหลดโมเดล: intfloat/multilingual-e5-small...")
bert_model = SentenceTransformer('intfloat/multilingual-e5-small')
print(f"✅ โหลดสำเร็จ! Vector dimension: {bert_model.get_sentence_embedding_dimension() or 384} มิติ")

⏳ กำลังโหลดโมเดล: intfloat/multilingual-e5-small...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11775.60it/s]


✅ โหลดสำเร็จ! Vector dimension: 384 มิติ


C:\Users\anand\AppData\Local\Temp\ipykernel_6624\4292999342.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ โหลดสำเร็จ! Vector dimension: {bert_model.get_sentence_embedding_dimension() or 384} มิติ")


## 🗄️ Step 5: Index ลง ChromaDB

In [45]:
import chromadb

chroma_client = chromadb.Client()
collection_name = "yuedpao_products_e5"

if collection_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(collection_name)

collection = chroma_client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})

print("⏳ Encoding embeddings (batch_size=32)...")
embeddings = bert_model.encode(documents, convert_to_tensor=False, batch_size=32, show_progress_bar=True).tolist()
collection.add(ids=doc_ids, documents=documents, embeddings=embeddings, metadatas=metadatas)
print(f"🎉 ChromaDB indexed {collection.count():,} documents!")

⏳ Encoding embeddings (batch_size=32)...


Batches: 100%|██████████| 22/22 [00:09<00:00,  2.30it/s]


🎉 ChromaDB indexed 695 documents!


## 🔤 Step 6: สร้าง BM25 Corpus

In [46]:
from rank_bm25 import BM25Okapi
from pythainlp.tokenize import word_tokenize

def bm25_tokenizer(text: str) -> List[str]:
    clean_doc = text.replace("passage: ", "")
    tokens = word_tokenize(clean_doc, engine="newmm")
    return [t.strip().lower() for t in tokens if t.strip()]

print("⏳ Tokenizing 695 documents for BM25...")
bm25_corpus = [bm25_tokenizer(doc) for doc in documents]
bm25_model = BM25Okapi(bm25_corpus)
print(f"✅ BM25 Index built! ({len(bm25_corpus):,} documents)")

⏳ Tokenizing 695 documents for BM25...
✅ BM25 Index built! (695 documents)


## 🔀 Step 7: ฟังก์ชัน RRF Hybrid Search
$$\text{RRF\_Score}(d) = \frac{1}{k + r_{\text{BM25}}} + \frac{1}{k + r_{\text{Vector}}} \quad (k=60)$$

In [47]:
def rrf_hybrid_search(user_query: str, top_k: int = 5, k_constant: int = 60):
    start_t = time.perf_counter()

    # BM25 ranks
    query_tokens = bm25_tokenizer(user_query)
    bm25_scores = bm25_model.get_scores(query_tokens)
    bm25_ranked_indices = np.argsort(bm25_scores)[::-1]

    # Vector ranks
    query_emb = bert_model.encode(f"query: {user_query}", convert_to_tensor=False).tolist()
    chroma_results = collection.query(query_embeddings=[query_emb], n_results=len(documents))
    vector_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(chroma_results["ids"][0])}

    # RRF fusion
    rrf_scores = {}
    for bm25_rank, idx in enumerate(bm25_ranked_indices):
        doc_id = doc_ids[idx]
        r_bm25 = bm25_rank + 1
        r_vec = vector_rank_map.get(doc_id, 9999)
        rrf_scores[doc_id] = {
            "score": (1.0 / (k_constant + r_bm25)) + (1.0 / (k_constant + r_vec)),
            "bm25_rank": r_bm25,
            "vector_rank": r_vec,
            "metadata": metadatas[idx]
        }

    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_k]
    latency_ms = (time.perf_counter() - start_t) * 1000.0
    return sorted_rrf, latency_ms

## 📋 Step 8: โหลด QA Benchmark Dataset จาก JSON (50 คำถาม)

In [48]:
qa_json_path = "qa_benchmark_50.json"
if not os.path.exists(qa_json_path):
    qa_json_path = os.path.join(os.path.dirname(os.path.abspath("")), "notebooks", "intent_rank", "qa_benchmark_50.json")

with open(qa_json_path, encoding="utf-8") as f:
    qa_dataset = json.load(f)

# แสดง distribution ของหมวดคำถาม
from collections import Counter
cat_counts = Counter(item["category"] for item in qa_dataset)

print(f"✅ โหลด QA Benchmark สำเร็จ! ทั้งหมด {len(qa_dataset)} คำถาม")
print("\n📊 สัดส่วนหมวดหมู่การทดสอบ (Category Distribution):")
for cat, cnt in sorted(cat_counts.items()):
    bar = "█" * cnt
    print(f"  {cat:<35} {bar} ({cnt} คำถาม)")

print("\n📋 ตัวอย่าง QA 5 รายการแรก:")
for item in qa_dataset[:5]:
    price_str = f" | max ฿{item['max_price']}" if item.get("max_price") else ""
    print(f"  [{item['id']}] {item['query'][:55]:<55} → expect: '{item['expected_keyword']}'{price_str}")

✅ โหลด QA Benchmark สำเร็จ! ทั้งหมด 50 คำถาม

📊 สัดส่วนหมวดหมู่การทดสอบ (Category Distribution):
  Exact Model & Color                 ██████████ (10 คำถาม)
  Natural Language Fabric Touch       ██████████ (10 คำถาม)
  Price Boundary                      ██████████ (10 คำถาม)
  Target Persona                      ██████████ (10 คำถาม)
  Typo Resilience                     ██████████ (10 คำถาม)

📋 ตัวอย่าง QA 5 รายการแรก:
  [QA-01] อยากได้เสื้อโปโล Running Roulette สี Dark Gray          → expect: 'Running Roulette'
  [QA-02] เสื้อยืดรุ่น Kodnum สี Black มีไหม                      → expect: 'Kodnum'
  [QA-03] เสื้อยืด Ultrasoft คอกลมสี Smoke Gray                   → expect: 'Smoke Gray'
  [QA-04] อยากได้ Running Roulette สีฟ้า                          → expect: 'Running Roulette'
  [QA-05] Ultrasoft V Neck สี Lavender                            → expect: 'Lavender'


## 📊 Step 9: รันประเมินผล QA Benchmark ครบ 50 คำถาม (Hit Rate@5 | MRR@5 | Latency)

In [49]:
hits_at_5   = 0
mrr_scores  = []
latencies   = []
detail_rows = []

for item in qa_dataset:
    query   = item["query"]
    exp_kw  = item["expected_keyword"].lower()
    max_p   = item.get("max_price")

    results, lat_ms = rrf_hybrid_search(query, top_k=5)
    latencies.append(lat_ms)

    found_rank = 0
    top1_name  = results[0][1]["metadata"]["name"] if results else "N/A"

    for rank, (doc_id, res) in enumerate(results):
        meta = res["metadata"]
        haystack = " | ".join([meta["name"], meta["category"], meta["fabric"]]).lower()
        is_match = exp_kw in haystack
        if max_p is not None:
            is_match = is_match and (meta["price"] <= max_p)
        if is_match:
            found_rank = rank + 1
            break

    if found_rank > 0:
        hits_at_5 += 1
        mrr_scores.append(1.0 / found_rank)
        status = f"✅ Rank #{found_rank}"
    else:
        mrr_scores.append(0.0)
        status = "❌ Miss"

    detail_rows.append({
        "id": item["id"], "category": item["category"],
        "query": query, "top1_name": top1_name,
        "found_rank": found_rank, "status": status, "latency_ms": lat_ms
    })

# ─── Print Detail Table ───────────────────────────────────────────────────────
COL = 100
print("=" * COL)
print(f"{'📊 QA Benchmark Evaluation Report — 50 Scenarios':^{COL}}")
print("=" * COL)
print(f"{'ID':<7} │ {'Category':<28} │ {'Query':<36} │ {'Top-1 Name':<22} │ {'Status'}")
print("-" * COL)

for row in detail_rows:
    q_short  = row["query"][:35]
    n_short  = row["top1_name"][:21]
    print(f"{row['id']:<7} │ {row['category']:<28} │ {q_short:<36} │ {n_short:<22} │ {row['status']}")

# ─── Per-category summary ─────────────────────────────────────────────────────
cat_stats: Dict[str, Dict] = {}
for row in detail_rows:
    c = row["category"]
    if c not in cat_stats:
        cat_stats[c] = {"total": 0, "hits": 0, "mrr_sum": 0.0}
    cat_stats[c]["total"] += 1
    if row["found_rank"] > 0:
        cat_stats[c]["hits"] += 1
        cat_stats[c]["mrr_sum"] += 1.0 / row["found_rank"]

print("=" * COL)
print(f"{'📈 Per-Category Breakdown':^{COL}}")
print("=" * COL)
print(f"{'Category':<35} {'Hit Rate@5':>11} {'MRR@5':>8} {'Count':>6}")
print("-" * COL)
for cat, s in sorted(cat_stats.items()):
    hr  = s["hits"] / s["total"] * 100
    mrr = s["mrr_sum"] / s["total"]
    print(f"{cat:<35} {hr:>10.1f}% {mrr:>8.4f} {s['total']:>6}")

# ─── Overall Summary ──────────────────────────────────────────────────────────
overall_hr  = hits_at_5 / len(qa_dataset) * 100
overall_mrr = np.mean(mrr_scores)
avg_lat     = np.mean(latencies)
p95_lat     = np.percentile(latencies, 95)

print("=" * COL)
print(f"{'🏆 Overall Performance Summary':^{COL}}")
print("=" * COL)
print(f"  • Total QA Scenarios  : {len(qa_dataset)} คำถาม")
print(f"  • Hit Rate@5          : {overall_hr:.2f}%  ({hits_at_5}/{len(qa_dataset)} ผ่าน)")
print(f"  • MRR@5               : {overall_mrr:.4f}")
print(f"  • Avg Latency         : {avg_lat:.2f} ms")
print(f"  • P95 Latency         : {p95_lat:.2f} ms")
print("=" * COL)

                          📊 QA Benchmark Evaluation Report — 50 Scenarios                           
ID      │ Category                     │ Query                                │ Top-1 Name             │ Status
----------------------------------------------------------------------------------------------------
QA-01   │ Exact Model & Color          │ อยากได้เสื้อโปโล Running Roulette ส  │ Running Roulette_Dark  │ ✅ Rank #1
QA-02   │ Exact Model & Color          │ เสื้อยืดรุ่น Kodnum สี Black มีไหม   │ Kodnum_Crop_Black      │ ✅ Rank #1
QA-03   │ Exact Model & Color          │ เสื้อยืด Ultrasoft คอกลมสี Smoke Gr  │ Ultrasoft Unisex Kid   │ ✅ Rank #1
QA-04   │ Exact Model & Color          │ อยากได้ Running Roulette สีฟ้า       │ Running Roulette_Ligh  │ ✅ Rank #1
QA-05   │ Exact Model & Color          │ Ultrasoft V Neck สี Lavender         │ Ultrasoft Unisex V Ne  │ ✅ Rank #1
QA-06   │ Exact Model & Color          │ เสื้อ Tailor Cool สีขาว              │ TAILOR COOL POLO สี D  │ ✅ Rank